# Data Creation Challenge Workflow

This notebook implements the end-to-end workflow for the Data Creation Challenge, transforming raw company IDs into actionable sentiment signals on the WorldQuant BRAIN platform.

### **Prerequisites**

Before running this notebook, please ensure you have the following:

1.  **Environment File**: Create a `.env` file in the same directory as this notebook (or in a parent directory) with your WorldQuant BRAIN credentials. It should look like this:
    ```
    BRAIN_EMAIL=your_email_here
    BRAIN_PASSWORD=your_password_here
    BRAIN_API_URL=https://api.worldquantbrain.com
    ```
2.  **Universe File**: Ensure `universe.json` is present in the directory.
3.  **Brain Session Module**: Ensure `brain_session.py` is present in the directory.

### **Workflow Overview**

1.  **Universe Definition**: 
    -   Load the target universe of ~3000 companies from `universe.json`.

2.  **Data Retrieval (Batch ID Approach)**: 
    -   Fetch daily news documents from the Data API.
    -   **Optimization**: Instead of fetching one-by-one, we batch IDs into chunks of 500. This reduces API calls from ~3,000/day to ~6/day, making it highly efficient, otherwise you may encounter api limit.

3.  **Sentiment Score Calculation**: 
    -   Process the raw text of news documents.
    -   Calculate a sentiment score for each entity mentioned, weighted by the relevance of the text chunk.

4.  **Aggregation**: 
    -   Consolidate individual entity scores into a single DataFrame (Rows: Dates, Columns: Entity IDs).

5.  **Upload to BRAIN**: 
    -   **MATRIX Data**: Convert daily scores into a single average value per entity (Float).
    -   **VECTOR Data**: Preserve the full list of scores for each entity (List of Floats).
    -   Upload both formats to BRAIN as data fields.

In [ ]:
import os
import json
import requests
from requests.auth import HTTPBasicAuth
import time
import io
import pyarrow
import pyarrow.parquet as parquet
import numpy as np
import pandas as pd
from datetime import date, timedelta, datetime
from collections import defaultdict
from typing import Union, List
from dotenv import load_dotenv

# Import BrainSession from separate file
from session import BrainSession

# Load environment variables
load_dotenv()

# --- Configuration ---

API_BASE_URL = "https://api.worldquantbrain.com"
EMAIL = os.getenv("BRAIN_EMAIL")
PASSWORD = os.getenv("BRAIN_PASSWORD")

session = BrainSession(
    base_url=API_BASE_URL,
    email=EMAIL,
    password=PASSWORD
)

# Fetch User ID dynamically
try:
    USER_ID = session.get("/users/self").json()["id"]
    print(f"Authenticated as User ID: {USER_ID}")
except Exception as e:
    print(f"Failed to fetch User ID: {e}")
    USER_ID = "UNKNOWN"

# Paths (Relative to this notebook)
UNIV_FILE = "universe.json"

# Rate Limiting Configuration
BATCH_SIZE = 500  # Number of company IDs per API request
REQUEST_DELAY = 1  # Delay in seconds between API requests
MAX_RETRIES = 10  # Maximum number of retries for failed requests
MAX_BACKOFF_TIME = 120  # Maximum wait time in seconds for exponential backoff

## Step 1: Universe Definition

Load the `universe.json` file to see the universe of companies.

In [ ]:
with open(UNIV_FILE, 'r') as f:
    top3000 = json.load(f)

print(f"Loaded {len(top3000)} companies from universe.")
print(f"Sample IDs: {top3000[:10]}")

## Step 2 & 3: Data Retrieval & Scoring (Streamlined)

Fetch daily news documents and calculate sentiment scores in-memory. We batch the 3000+ IDs into chunks of 500 and query the API for each chunk per day. This avoids the need for external category mapping files and reduces I/O.

In [ ]:
def get_daily_documents(target_date_str: str, company_ids: List[str]):
    start_timestamp = f"{target_date_str}T00:00:00.000Z"
    end_timestamp = f"{target_date_str}T23:59:59.999Z"
    data_endpoint = "/bigdata/v1/search"

    def make_request_with_retry(ids, batch_num, total_batches, max_retries=MAX_RETRIES):
        """Make request with exponential backoff retry logic for rate limiting"""
        ## there are many other api points and parameters can be loaded into this payload. please be creative
        payload = {
            "query": {
                "filters": {
                    "timestamp": {"start": start_timestamp, "end": end_timestamp},
                    "entity": {"any_of": ids}
                }
            }
        }
        
        for attempt in range(max_retries):
            response = None
            try:
                response = session.post(data_endpoint, json=payload)
                response.raise_for_status()
                return response.json().get('results', [])
            except requests.exceptions.HTTPError as http_err:
                # Use the response from the exception if available, or the local variable
                err_response = getattr(http_err, 'response', response)
                
                if err_response is not None and err_response.status_code == 429:  # Too Many Requests
                    # Exponential backoff: 2^attempt seconds (1, 2, 4, 8, 16 seconds)
                    wait_time = min(2 ** attempt, MAX_BACKOFF_TIME)  # Cap at max backoff time
                    print(f"  Rate limit hit for batch {batch_num}/{total_batches}. Waiting {wait_time}s before retry {attempt+1}/{max_retries}...")
                    time.sleep(wait_time)
                else:
                    print(f"  HTTP error for batch {batch_num}/{total_batches}: {http_err}")
                    break
            except Exception as req_err:
                print(f"  Request error for batch {batch_num}/{total_batches}: {req_err}")
                break
        
        print(f"  Failed to fetch batch {batch_num}/{total_batches} after {max_retries} attempts")
        return []

    # Batch IDs into chunks
    all_docs = []
    total_batches = (len(company_ids) + BATCH_SIZE - 1) // BATCH_SIZE
    
    print(f"  Processing {len(company_ids)} companies in {total_batches} batches...")
    
    for i in range(0, len(company_ids), BATCH_SIZE):
        batch_num = (i // BATCH_SIZE) + 1
        chunk = company_ids[i:i + BATCH_SIZE]
        
        # Add a small delay between requests to avoid hitting rate limits
        if batch_num > 1:
            time.sleep(REQUEST_DELAY)  # Delay between requests
        
        print(f"  Fetching batch {batch_num}/{total_batches} ({len(chunk)} companies)...")
        batch_docs = make_request_with_retry(chunk, batch_num, total_batches)
        all_docs.extend(batch_docs)
        
    print(f"  Completed: Retrieved {len(all_docs)} documents for {target_date_str}")
    return all_docs

In [ ]:
def calculate_scores_for_day(documents, valid_ids_set):
    """Process raw documents and return a dict of {entity_id: [scores]}"""
    day_scores = defaultdict(list)
    
    for article in documents:
        chunk_scores = []
        article_entities = set()

        # 1. Calculate average score for the article based on chunks
        for chunk in article.get('chunks', []):
            relevance = chunk.get('relevance', 0)
            sentiment = chunk.get('sentiment', 0)
            if relevance is not None and sentiment is not None:
                score = relevance * sentiment
                chunk_scores.append(score)
            
            for detection in chunk.get('detections', []):
                if detection.get('type') == 'entity':
                    entity_id = detection.get('id')
                    if entity_id and entity_id in valid_ids_set:
                        article_entities.add(entity_id)
        
        # 2. Assign article score to all found entities
        if chunk_scores and article_entities:
            article_score = round((sum(chunk_scores) / len(chunk_scores)) * 100, 2)
            for entity_id in article_entities:
                day_scores[entity_id].append(article_score)
                
    return day_scores

In [ ]:
# Execution Logic - Fetch and Process In-Memory

def run_workflow(start_date_input: Union[str, date, datetime], end_date_input: Union[str, date, datetime], universe_ids: List[str]):
    """
    Executes the data retrieval and scoring workflow for a given date range.
    
    Args:
        start_date_input (str, date, datetime): Start date (e.g., '2021-01-01' or date object).
        end_date_input (str, date, datetime): End date (e.g., '2021-01-07' or date object).
        universe_ids (list): List of company IDs to process.
        
    Returns:
        dict: Aggregated data structure { entity_id: { date_str: [scores] } }
    """
    
    # Helper to parse date inputs
    def parse_date(d):
        if isinstance(d, str):
            return datetime.strptime(d, "%Y-%m-%d").date()
        elif isinstance(d, datetime):
            return d.date()
        elif isinstance(d, date):
            return d
        else:
            raise ValueError(f"Invalid date format: {d} (type: {type(d)})")

    start_date = parse_date(start_date_input)
    end_date = parse_date(end_date_input)
    
    start_date_str = start_date.strftime("%Y-%m-%d")
    end_date_str = end_date.strftime("%Y-%m-%d")

    print(f"--- Starting daily document retrieval for {len(universe_ids)} companies ---")
    print(f"--- Period: {start_date_str} to {end_date_str} ---")

    delta = timedelta(days=1)
    valid_ids_set = set(universe_ids)
    
    # Structure: { entity_id: { date_str: [scores] } }
    all_entity_data = defaultdict(lambda: defaultdict(list))

    current_date = start_date
    while current_date <= end_date:
        date_str = current_date.strftime("%Y-%m-%d")
        print(f"Processing data for {date_str}...")
        
        # 1. Fetch
        documents = get_daily_documents(date_str, universe_ids)
        
        # 2. Score
        daily_scores = calculate_scores_for_day(documents, valid_ids_set)
        
        # 3. Aggregate into main structure
        for entity_id, scores in daily_scores.items():
            all_entity_data[entity_id][date_str] = scores
            
        current_date += delta

    print("\n--- Data processing complete ---")
    return all_entity_data

# Run the workflow
# Example usage with string inputs (can also be date/datetime objects)
all_entity_data = run_workflow(start_date_input="2021-01-01", end_date_input="2021-01-01", universe_ids=top3000)

## Step 4: DataFrame Construction

Convert the nested dictionary structure into a pandas DataFrame ready for upload.

In [ ]:
def create_dataframe(entity_data, start_date_input, end_date_input, universe_ids, output_path="sentiment_data.parquet"):
    """
    Converts the aggregated entity data into a pandas DataFrame and saves it as a parquet file.
    """
    # Create a list of all dates covered
    # pd.date_range handles various input types gracefully
    date_range = pd.date_range(start=start_date_input, end=end_date_input, freq='D')
    sorted_dates = [d.strftime("%Y-%m-%d") for d in date_range]

    # Construct DataFrame
    # Rows: Dates
    # Columns: Entity IDs
    # Values: List of scores (or empty list/None)

    df_data = {}
    for entity_id in universe_ids:
        # Get scores for each date, default to None if no scores found
        entity_daily_scores = entity_data.get(entity_id, {})
        df_data[entity_id] = [entity_daily_scores.get(d, None) for d in sorted_dates]

    df_raw = pd.DataFrame(df_data, index=pd.to_datetime(sorted_dates))
    df_raw.index.name = 'date'
    
    # Save to parquet locally
    df_raw.to_parquet(output_path)
    print(f"DataFrame saved to {output_path}")
    
    return df_raw

# Create the DataFrame
# Re-using the same date inputs as the workflow run
df_filtered = create_dataframe(all_entity_data, "2021-01-01", "2021-01-01", top3000)
print(f"Created DataFrame with shape: {df_filtered.shape}")

## Step 5: Upload to BRAIN

Upload the processed sentiment data to the WorldQuant BRAIN platform as `MATRIX` and `VECTOR` data fields.

In [ ]:
#### If you want to create a "MATRIX" TYPE Data
def calc_average(scores):
    if isinstance(scores, list) and scores:
        return sum(scores) / len(scores)
    return 0.0 # Default value for missing data

def create_data_field(data_field_id, data_frame, region, delay, type_name, source):
    data_frame_copy = data_frame.copy()
    buffer = io.BytesIO()
    table = pyarrow.Table.from_pandas(data_frame_copy, preserve_index=True)
    parquet.write_table(table, buffer, compression='GZIP')
    buffer.seek(0)

    upload_url = "/users/self/data-fields"
    
    data = {
        'field': json.dumps({
            'id': data_field_id, 
            'type': type_name, 
            'source': source, 
            'data': [{'region': region, 'delay': delay}]
        })
    }
    
    files = {'data': buffer}
    
    print(f"Uploading data field '{data_field_id}'...")
    try:
        response = session.post(upload_url, data=data, files=files)
        if response.status_code == 201:
            print("Data field created successfully.")
        else:
            print(f"Error creating data field: {response.status_code}")
            print(response.text)
        response.raise_for_status()
    except Exception as e:
        print(f"Upload failed: {e}")

# ---------------------------------------------------------
# 1. MATRIX Upload (Averaged Scores)
# ---------------------------------------------------------
print("\n--- Processing MATRIX Data (Averaged Scores) ---")

# To create a MATRIX data field, we need a DataFrame where each cell contains a single float value.
df_matrix = df_filtered.copy()
for col in df_matrix.columns:
    df_matrix[col] = df_matrix[col].apply(calc_average)

# Ensure all data is float
df_matrix = df_matrix.astype(float, errors='ignore')

create_data_field(
    data_field_id=f"{USER_ID}_sentiment_scores_matrix", 
    data_frame=df_matrix,
    region='USA',
    delay=1,
    type_name='MATRIX',
    source='BigData'
)

#### If you want to create a "VECTOR" TYPE Data
def parse_keep_as_list(scores):
    if isinstance(scores, list):
        return [float(s) for s in scores if s is not None]
    return []

# ---------------------------------------------------------
# 2. VECTOR Upload (List of Scores)
# ---------------------------------------------------------
print("\n--- Processing VECTOR Data (List of Scores) ---")

# To create a VECTOR data field, we need a DataFrame where each cell contains a list of floats.
df_vector = df_filtered.copy()
for col in df_vector.columns:
    df_vector[col] = df_vector[col].apply(parse_keep_as_list)

create_data_field(
    data_field_id=f"{USER_ID}_sentiment_scores_vector", 
    data_frame=df_vector,
    region='USA',
    delay=1,
    type_name='VECTOR',
    source='BigData'
)

## Step 6: Go to the platform and simulate the Alpha

Remember to Tag your Alphas and ONLY 'created fields' plus support fields are allowed to be used in your Alphas.